In [29]:
from datetime import datetime

import pandas as pd
import os
import clean_tweets as ct
from model import *

In [30]:
datasets_folder = "/Users/alalousis/PycharmProjects/AI_detector/datasets"
results_folder = "/Users/alalousis/PycharmProjects/AI_detector/results"
real_tweets_filename = "real_dataset.csv"
ai_tweets_filename = "ai_dataset_10000.csv"

cols = [9]
real_tweets_df = pd.read_csv(os.path.join(datasets_folder, real_tweets_filename), encoding="UTF-8", header=0, usecols=cols)
real_tweets_df["created_by"] = "human"
real_tweets_df.head()
real_tweets_df.info()

cols = [1]
ai_tweets_df = pd.read_csv(os.path.join(datasets_folder, ai_tweets_filename), encoding="UTF-8", header=0, usecols=cols)
ai_tweets_df["created_by"] = "bot"
ai_tweets_df.head()
ai_tweets_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238646 entries, 0 to 238645
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   text        238646 non-null  object
 1   created_by  238646 non-null  object
dtypes: object(2)
memory usage: 3.6+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        10000 non-null  object
 1   created_by  10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB


In [31]:
ai_tweets_df['text'] = ai_tweets_df['text'].str.strip('"')

In [32]:
main_dataset = pd.concat([real_tweets_df[:8000], ai_tweets_df[:2000]], ignore_index=True)

In [33]:
main_dataset = main_dataset.sample(frac=1).reset_index(drop=True)

In [35]:
clean = ct.clean_tweets(main_dataset)

In [37]:
import numpy as np

texts = list(main_dataset["text"])
L = [len(text) for text in texts]
NL = np.array(L)
# Compute the mean and the standard deviation of the sequence lengths.
Lmean,Lstd = NL.mean(),NL.std()
# labels = main_dataset["created_by"]
main_dataset['label'] = main_dataset['created_by'].apply(lambda x: 0 if x == 'human' else 1)
main_dataset.drop(columns=['created_by'], inplace=True)
labels = main_dataset["label"].to_list()

In [39]:
Labels = np.array(labels)
Labels_min = Labels.min()
Labels_max = Labels.max()
Labels_bins = np.arange(Labels_min,Labels_max+2)
LabelsHist,_ = np.histogram(Labels,Labels_bins)
class_priors = LabelsHist / LabelsHist.sum()

In [41]:
from transformers import BertTokenizer, BertModel

model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertModel.from_pretrained(model_name)
max_length = int(Lmean+3*Lstd)+1

In [42]:
ground_truth_dataset = ct.SentimentDataset(texts,labels,tokenizer,max_length)

In [44]:
retain_percent = 0.01
train_percent = 0.80
epochs = 5
batch_size = 2
freeze_bert = False
batch_save_period = 5
classes_num = len(set(labels))

In [45]:
from torch.utils.data import random_split, DataLoader

# Set the train and the test sizes.
train_size = int(train_percent * len(ground_truth_dataset))
test_size = len(ground_truth_dataset) - train_size

# Split the ground truth dataset into training and testing subsets.
train_dataset, test_dataset = random_split(
    ground_truth_dataset, [train_size,test_size])

# Instantiate the training and testing data loaders.
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=batch_size)


In [46]:
# -----------------------------------------------------------------------------
#    CLASSIFIER, LOSS FUNCTION, OPTIMIZER AND CHECK POINT INITIALIZATION:
# -----------------------------------------------------------------------------
# Set the device on which training will be performed.
device = get_execution_device()
model = SentimentClassifier(bert_model,classes_num,freeze_bert=freeze_bert)
optimizer = torch.optim.Adam(model.parameters(),lr=2e-5)
# Set the class weights to be inversly proportinal to the class priors.
class_weights = 1 / class_priors
# Convert class weights to tensor
weight = torch.tensor(class_weights, dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weight)
# Move the initialized model to the appropriate device.
model = model.to(device)

MPS GPU is available!


In [47]:
# Define the name of the checkpoint directory
checkpoint_directory = f"checkpoints_{retain_percent}_{train_percent}_{epochs}_{batch_size}_{freeze_bert}_{batch_save_period}"
# Define the name of the file that will actually store the checkpoint dictionary.
checkpoint_filename = "model_checkpoint.pth"
# Create the full path for the checkpoint directory within the current directory
checkpoint_path = os.path.join(checkpoint_directory,checkpoint_filename)
# Set the number of batches within each training epoch after which a checkpoint
# entry will be saved.
batch_save_period = 5
# Create the checkpoint directory and the corresponding file in case it does
# not exist. In that case, create the file and save the initial state of the
# training process.
if not os.path.exists(checkpoint_directory):
    os.makedirs(checkpoint_directory)
    f = open(checkpoint_path,"w")
    f.close()
    # Set the initial state variables of the training process.
    epoch = 0
    batch_count=0
    model_state_dict = model.state_dict()
    optimizer_state_dict = optimizer.state_dict()
    train_accuracy = 0
    test_accuracy = 0
    torch.save({
        'epoch':epoch,
        'batch_count':batch_count,
        'model_state_dict':model.state_dict(),
        'optimizer_state_dict':optimizer.state_dict(),
        'train_accuracy':train_accuracy,
        'test_accuracy':test_accuracy
        },checkpoint_path)

In [48]:
# Call the model training method.
train_model(model,train_loader,test_loader,optimizer,loss_fn,epochs,
            checkpoint_path=checkpoint_path,batch_save_period=batch_save_period, device=device)

Epoch: 5 / 5 train_acc: 0.968625 test_acc: 0.9715: 100%|██████████| 4000/4000 [32:57<00:00,  2.02it/s]
100%|██████████| 1000/1000 [01:03<00:00, 15.73it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/Users/alalousis/PycharmProjects/AI_detector/history/checkpoint_2025-12-01 21:00:18.295890.txt'